In [1]:
DATA = '../data/'
FIGS = '../results/figures/'
CACHE = '../f1_cache'

In [2]:
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_pickle(DATA + 'laps_clean.pkl') # load the clean lap data
print(f"loaded {len(df)} laps, {df['RaceID'].nunique()} races, {df['Circuit'].nunique()} circuits")

# Hold out the demo race entirely
demo = df[(df['Year'] == 2025) & (df['Circuit'] == 'Barcelona')]
assert len(demo) > 0, "demo filter matched nothing - check df['Circuit'].unique()"
pool = df.drop(demo.index)
print(f"demo race: {len(demo)} laps, training pool: {len(pool)} laps")

# Split BY RACE WEEKEND, never randomly by lap
races = pool['RaceID'].unique()
rng = np.random.default_rng(0)
test_races = rng.choice(races, size=int(len(races) * 0.2), replace=False) # randomly selected 20% of races for testing

train = pool[~pool['RaceID'].isin(test_races)] # the pile of laps used for training the model
test = pool[pool['RaceID'].isin(test_races)] # the pile of laps used for scoring the model's accuracy
print(f"train: {len(train)} laps / test: {len(test)} laps")

# --- Tier 1: heuristic ---
ref = (train.groupby(['Circuit', 'Driver'])['LapSeconds']
            .median().rename('pred1').reset_index())
t1 = test.merge(ref, on=['Circuit', 'Driver'], how='inner')
mae1 = mean_absolute_error(t1['LapSeconds'], t1['pred1'])

# The inner join drops test laps whose (Circuit, Driver) pair never appeared in training.
# Tier 2 must be scored on the SAME laps or the comparison is meaningless.
keep = test.set_index(['Circuit', 'Driver']).index.isin(
    ref.set_index(['Circuit', 'Driver']).index)
print(f"tier 1 scored on {keep.sum()} of {len(test)} test laps")

# --- Tier 2: linear regression ---
AGE_COLS  = sorted(c for c in df.columns if c.startswith('age_'))
FUEL_COLS = sorted(c for c in df.columns if c.startswith('fuel_'))    # NEW

feat = ['AirTemp', 'TrackTemp'] + AGE_COLS + FUEL_COLS
cats = ['Compound', 'Circuit', 'Driver', 'FreshTyre', 'TeamYear']

# drop_first drops the first category PRESENT IN THE FRAME YOU PASS. Call it separately on
# train, test and demo and the three calls drop different categories, so reindex silently
# encodes rows as the wrong one. On demo - a single race - every categorical has one value,
# so drop_first deletes it entirely and every lap is predicted at the baseline circuit.
# Pinning the category list to train makes the columns match by construction.
def encode(d):
    d = d[feat + cats].copy()
    for c in cats:
        d[c] = pd.Categorical(d[c], categories=sorted(train[c].dropna().unique()))
    return pd.get_dummies(d, columns=cats, drop_first=True)

Xtr = encode(train) # builds the training input table
Xte = encode(test) # builds the test input table - same columns by construction, no reindex needed

lin = LinearRegression().fit(Xtr, train['LapSeconds'])
pred2 = lin.predict(Xte)
mae2 = mean_absolute_error(test['LapSeconds'], pred2) # on all test laps
mae2_fair = mean_absolute_error(test['LapSeconds'][keep], pred2[keep]) # on tier 1's laps

print(f"\nTier 1 (heuristic): {mae1:.3f} s")
print(f"Tier 2 (linear):    {mae2_fair:.3f} s  <- same laps as tier 1, this is the comparison")
print(f"improvement:        {mae1 - mae2_fair:.3f} s")
print(f"Tier 2, all laps:   {mae2:.3f} s  <- use this one against tier 3")

coef = dict(zip(Xtr.columns, lin.coef_))
CIRCUIT_AGE = [c for c in AGE_COLS if c not in ('age_demo_soft', 'age_demo_med')]
#print(f"\ntyre-age slope, median over {len(CIRCUIT_AGE)} circuits: {np.median([coef[c] for c in CIRCUIT_AGE]):+.4f} s/lap")
#print(f"  SoftAge (global): {coef['SoftAge']:+.4f}   MedAge (global): {coef['MedAge']:+.4f}")
#print(f"LapNumber coefficient: {coef['LapNumber']:+.4f} s per lap of race")

b = df[(df['Circuit'] == 'Barcelona') & (df['Year'] != 2025)]
X = pd.get_dummies(b[['TyreLife', 'LapNumber', 'AirTemp', 'TrackTemp', 'Compound', 'Driver', 'Year']],
                   columns=['Compound', 'Driver', 'Year'], drop_first=True)
for c in ['MEDIUM', 'SOFT']:
    X[f'{c}_x_age'] = X.get(f'Compound_{c}', 0) * b['TyreLife'].values
m = LinearRegression().fit(X, b['LapSeconds'])
d = dict(zip(X.columns, m.coef_))
t = d['TyreLife']
print(f"HARD {t:+.4f}   MEDIUM {t + d['MEDIUM_x_age']:+.4f}   SOFT {t + d['SOFT_x_age']:+.4f}")
print("target, Barcelona 2023+24:  HARD +0.0585  MEDIUM +0.0619  SOFT +0.0667")

# Tier 2 on the demo race. Step 5 prints a DEMO RACE number too, but that one belongs to
# Tier 3. This is the model that ships, on the one race the app replays - so of every
# number in this notebook, this is the one the app is actually judged on.
print(f"\nTier 2, DEMO RACE: {mean_absolute_error(demo['LapSeconds'], lin.predict(encode(demo))):.3f} s")

b = coef['age_Barcelona']
print(f"\nBarcelona HARD:   {b:+.4f}")
print(f"Barcelona MEDIUM: {b + coef['age_demo_med']:+.4f}")
print(f"Barcelona SOFT:   {b + coef['age_demo_soft']:+.4f}")
print(f"\nBarcelona fuel:   {coef['fuel_Barcelona']:+.4f} s per lap of race")
print(f"fuel, median over circuits: {np.median([coef[c] for c in FUEL_COLS]):+.4f}")

raw = pd.read_pickle(DATA + 'laps_raw.pkl')
b = raw[(raw['Year'] == 2025) & (raw['Circuit'] == 'Barcelona')].copy()
b['s'] = b['LapTime'].dt.total_seconds()
b['med'] = b.groupby('Driver')['s'].transform('median')
io = b[b['PitInTime'].notna() | b['PitOutTime'].notna()]
print((io['s'] - io['med']).describe())

loaded 74015 laps, 83 races, 24 circuits
demo race: 951 laps, training pool: 73064 laps
train: 59243 laps / test: 13821 laps
tier 1 scored on 12258 of 13821 test laps

Tier 1 (heuristic): 1.620 s
Tier 2 (linear):    1.296 s  <- same laps as tier 1, this is the comparison
improvement:        0.324 s
Tier 2, all laps:   1.342 s  <- use this one against tier 3
HARD +0.0578   MEDIUM +0.0638   SOFT +0.0665
target, Barcelona 2023+24:  HARD +0.0585  MEDIUM +0.0619  SOFT +0.0667

Tier 2, DEMO RACE: 0.905 s

Barcelona HARD:   +0.0520
Barcelona MEDIUM: +0.0625
Barcelona SOFT:   +0.0698

Barcelona fuel:   -0.0516 s per lap of race
fuel, median over circuits: -0.0549
count    109.000000
mean      17.563133
std       15.327575
min        2.544000
25%        5.137000
50%       16.575500
75%       19.008000
max       68.561500
dtype: float64


/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279:

In [3]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error

num = ['TyreLife', 'LapNumber', 'AirTemp', 'TrackTemp', 'SoftAge', 'MedAge'] + AGE_COLS
cat = ['Compound', 'Circuit', 'Driver', 'Team', 'FreshTyre', 'Year']

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
enc.fit(train[cat].astype(str))

def prep(d):
    X = d[num].reset_index(drop=True).copy()
    codes = pd.DataFrame(enc.transform(d[cat].astype(str)),
                         columns=cat)
    return pd.concat([X, codes], axis=1)

# The encoder emits -1 for a category never seen in training. HistGradientBoosting
# expects non-negative codes, so check whether -1 ever actually occurs before trusting it.
print("unseen categories in test:", int((prep(test)[cat] == -1).sum().sum()))
print("unseen categories in demo:", int((prep(demo)[cat] == -1).sum().sum()))

cat_idx = list(range(len(num), len(num) + len(cat)))

model = HistGradientBoostingRegressor(
    categorical_features=cat_idx,
    early_stopping=False, # its internal validation split is by lap, which leaks across race weekends
    random_state=0
).fit(prep(train), train['LapSeconds'])

mae_train = mean_absolute_error(train['LapSeconds'], model.predict(prep(train)))
mae3      = mean_absolute_error(test['LapSeconds'],  model.predict(prep(test)))
mae_demo  = mean_absolute_error(demo['LapSeconds'],  model.predict(prep(demo)))

print(f"Tier 1 (heuristic):   {mae1:.3f} s")
print(f"Tier 2 (linear):      {mae2:.3f} s")
print(f"Tier 3 (boosted):     {mae3:.3f} s")
print(f"  training error:     {mae_train:.3f} s")
print(f"  DEMO RACE:          {mae_demo:.3f} s")

# Which laps carry the error: unseen categories, or particular races
err = abs(model.predict(prep(test)) - test['LapSeconds'].values)
u = (prep(test)[cat] == -1).any(axis=1).values
print(f"\ntest laps: {len(test)}, laps with an unseen category: {u.sum()}")
print(f"MAE, clean laps:  {err[~u].mean():.3f} s")
print(f"MAE, unseen laps: {err[u].mean():.3f} s" if u.sum() else "no unseen laps")
print(test.assign(err=err).groupby('RaceID')['err'].agg(['mean', 'count']).sort_values('mean'))

# Tier 2 vs Tier 3 on the same races. The paired difference cancels race-to-race noise.
d = (test.assign(e2=abs(pred2 - test['LapSeconds'].values), e3=err)
         .groupby('RaceID')[['e2', 'e3']].mean())
print(d.assign(diff=d['e3'] - d['e2']).sort_values('diff'))
print(f"\nTier 3 beat Tier 2 in {(d['e3'] < d['e2']).sum()} of {len(d)} races")

print({k: round(v, 3) for k, v in coef.items() if k.startswith('Year')})

unseen categories in test: 0
unseen categories in demo: 0
Tier 1 (heuristic):   1.620 s
Tier 2 (linear):      1.342 s
Tier 3 (boosted):     1.853 s
  training error:     0.472 s
  DEMO RACE:          1.938 s

test laps: 13821, laps with an unseen category: 0
MAE, clean laps:  1.853 s
no unseen laps
             mean  count
RaceID                  
2024_3   0.500445    812
2022_15  0.620127   1024
2023_1   0.620687    849
2024_17  0.778193    828
2024_19  0.934096    860
2022_10  1.108480    630
2025_4   1.458029    914
2024_6   1.483003    905
2024_8   1.735037   1159
2024_12  1.874292    626
2023_5   1.928782   1043
2023_9   3.005553   1133
2022_4   3.009544    712
2025_2   3.074055    914
2022_11  3.130476    782
2023_12  5.119049    630
               e2        e3      diff
RaceID                               
2025_2   4.543905  3.074055 -1.469850
2022_15  1.894746  0.620127 -1.274618
2024_17  1.533917  0.778193 -0.755724
2024_19  1.076316  0.934096 -0.142220
2024_3   0.591162  0.5

In [4]:
demo_pred = lin.predict(encode(demo)) # Tier 2 - the model that ships. NOT Tier 3's model/prep
comp = demo.reset_index(drop=True).copy()
comp['Pred'] = demo_pred

per_driver = comp.groupby('Driver').agg(
    laps=('LapSeconds', 'size'),
    actual=('LapSeconds', 'sum'),
    predicted=('Pred', 'sum'),
)
per_driver['delta'] = per_driver['predicted'] - per_driver['actual']
per_driver['delta_per_lap'] = per_driver['delta'] / per_driver['laps']

print(per_driver.sort_values('delta').to_string())
print(f"\nmedian absolute total delta: "
      f"{per_driver['delta'].abs().median():.1f} s")

print(train.groupby('Driver').size().sort_values().head(10))

        laps    actual    predicted      delta  delta_per_lap
Driver                                                       
COL       50  4110.803  4041.482825 -69.320175      -1.386403
SAI       51  4151.827  4087.786545 -64.040455      -1.255695
OCO       51  4169.980  4107.132229 -62.847771      -1.232309
BEA       51  4160.241  4098.722763 -61.518237      -1.206240
ALO       54  4399.400  4339.331927 -60.068073      -1.112372
LAW       52  4237.961  4181.494937 -56.466063      -1.085886
BOR       52  4254.390  4204.176333 -50.213667      -0.965647
TSU       48  3899.753  3856.859722 -42.893278      -0.893610
LEC       53  4248.396  4208.365762 -40.030238      -0.755288
GAS       53  4316.690  4280.044259 -36.645741      -0.691429
NOR       53  4227.223  4192.048884 -35.174116      -0.663663
ANT       47  3809.963  3775.964334 -33.998666      -0.723376
RUS       53  4260.329  4227.462593 -32.866407      -0.620121
VER       51  4063.860  4033.871365 -29.988635      -0.588012
HAD     

/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [5]:
demo_pred = lin.predict(encode(demo)) # Tier 2 - the model that ships. NOT Tier 3's model/prep
comp = demo.reset_index(drop=True).copy()
comp['Pred'] = demo_pred

per_driver = comp.groupby('Driver').agg(
    laps=('LapSeconds', 'size'),
    actual=('LapSeconds', 'sum'),
    predicted=('Pred', 'sum'),
)
per_driver['delta'] = per_driver['predicted'] - per_driver['actual']
per_driver['delta_per_lap'] = per_driver['delta'] / per_driver['laps']

print(per_driver.sort_values('delta').to_string())
print(f"\nmedian absolute total delta: "
      f"{per_driver['delta'].abs().median():.1f} s")

DEMO_CIRCUIT = 'Barcelona'

def derive(d):
    """Rebuild every derived column from Circuit / Compound / TyreLife / LapNumber.
       Anything that changes TyreLife or Compound MUST pass through here afterwards."""
    d = d.copy()
    for c in AGE_COLS + FUEL_COLS:
        d[c] = 0.0
    at = (d['Circuit'] == DEMO_CIRCUIT).values
    d[f'age_{DEMO_CIRCUIT}']  = at * d['TyreLife']
    d[f'fuel_{DEMO_CIRCUIT}'] = at * d['LapNumber']
    d['age_demo_soft'] = (at & (d['Compound'] == 'SOFT').values)   * d['TyreLife']
    d['age_demo_med']  = (at & (d['Compound'] == 'MEDIUM').values) * d['TyreLife']
    return d

# The check that makes it trustworthy: rebuilding the REAL race must reproduce exactly
# what Step 3 built. If this passes, hand-built laps are encoded the way the model expects.
chk = derive(demo)
for c in AGE_COLS + FUEL_COLS:
    assert np.allclose(chk[c].astype(float), demo[c].astype(float)), f"mismatch: {c}"
print("derive() reproduces Step 3 on the real race")

raw = pd.read_pickle(DATA + 'laps_raw.pkl')
b = raw[(raw['Year'] == 2025) & (raw['Circuit'] == DEMO_CIRCUIT)].copy()
b['s'] = b['LapTime'].dt.total_seconds()
b['med'] = b.groupby('Driver')['s'].transform('median')

inlap  = b[b['PitInTime'].notna()]
outlap = b[b['PitOutTime'].notna()]
pit_in  = (inlap['s']  - inlap['med']).median()
pit_out = (outlap['s'] - outlap['med']).median()
print(f"in-lap  {pit_in:.2f} s  (n={len(inlap)})")
print(f"out-lap {pit_out:.2f} s  (n={len(outlap)})")
print(f"PIT_LOSS = {pit_in + pit_out:.2f} s")

        laps    actual    predicted      delta  delta_per_lap
Driver                                                       
COL       50  4110.803  4041.482825 -69.320175      -1.386403
SAI       51  4151.827  4087.786545 -64.040455      -1.255695
OCO       51  4169.980  4107.132229 -62.847771      -1.232309
BEA       51  4160.241  4098.722763 -61.518237      -1.206240
ALO       54  4399.400  4339.331927 -60.068073      -1.112372
LAW       52  4237.961  4181.494937 -56.466063      -1.085886
BOR       52  4254.390  4204.176333 -50.213667      -0.965647
TSU       48  3899.753  3856.859722 -42.893278      -0.893610
LEC       53  4248.396  4208.365762 -40.030238      -0.755288
GAS       53  4316.690  4280.044259 -36.645741      -0.691429
NOR       53  4227.223  4192.048884 -35.174116      -0.663663
ANT       47  3809.963  3775.964334 -33.998666      -0.723376
RUS       53  4260.329  4227.462593 -32.866407      -0.620121
VER       51  4063.860  4033.871365 -29.988635      -0.588012
HAD     

/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/huytran/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [11]:
import re

out = pd.concat([train, test])[feat + cats + ['LapSeconds']].copy() # Tier 2's feature set - the model that ships
dem = demo[feat + cats + ['LapSeconds']].copy() # the held-out race, for Create ML's TESTING slot

# Circuit names carry spaces, hyphens and accents - 'age_Marina Bay', 'age_Montreal' -
# which Xcode cannot turn into Swift property names. Rename AFTER the frames are built
# (before, and you are renaming a leftover from the last run), and identically for both,
# or the testing file will not match the model's inputs.
safe = lambda c: re.sub(r'[^0-9A-Za-z_]', '_', c)
out.columns = [safe(c) for c in out.columns]
dem.columns = [safe(c) for c in dem.columns]

out['FreshTyre'] = out['FreshTyre'].astype(str)
dem['FreshTyre'] = dem['FreshTyre'].astype(str)

# No dtype fixing needed: TeamYear is already text ("Ferrari_2024"), so Create ML reads it
# as a category and gives each team-season its own offset, matching get_dummies.
out.to_csv(DATA + 'laptimes.csv', index=False)
dem.to_csv(DATA + 'demo.csv', index=False)
print(out.shape, dem.shape)
print(out.dtypes.value_counts())

a = pd.read_csv(DATA + 'laptimes.csv', nrows=5000)
b = pd.read_csv(DATA + 'demo.csv')

print("same names, same order:", list(a.columns) == list(b.columns))
print("only in one:", set(a.columns) ^ set(b.columns))

d = a.dtypes != b.dtypes
print("dtype mismatches:", int(d.sum()))
if d.any():
    print(pd.DataFrame({'laptimes': a.dtypes[d], 'demo': b.dtypes[d]}))

(73064, 58) (951, 58)
float64    53
object      5
Name: count, dtype: int64
same names, same order: True
only in one: set()
dtype mismatches: 0
